# Lab | LLMs through an API
## A support assistant for the bank

Over the last two days you routed the bank's customer messages with semantic search and with a fine-tuned DistilBERT. Both needed hundreds of labelled messages. The product team now asks a different question: **could a large language model do the job with no training at all, just a good prompt?** And while it is at it, could it extract the details of each message and draft the reply?

You will build that assistant with the OpenAI API on 60 real messages from the ten intents of yesterday's lab, measure its accuracy against yesterday's models, and keep track of what every step costs.

**The data** is a sample of the Banking77 dataset: `data/messages.csv` has the 60 test messages (6 per intent), and `data/examples.csv` has 20 other labelled messages (2 per intent) that you can use as examples in prompts.

**This lab practises** calling an LLM through an API. You will use, from today's **LLMs through an API** lesson: messages and roles, the system prompt, `temperature`, the `usage` field and the cost per token, zero-shot and few-shot prompts, structured outputs with Pydantic, and conversation history.

**By the end, you can:**

1. Implement an LLM-based classifier and extractor with prompts and structured outputs, and measure its accuracy and cost.
2. Explain when a prompt is enough and when a trained model is the better choice.

Each challenge ends with a **checkpoint** that tells you what you should see. The model's answers change a little from run to run, so your numbers can differ by a few points. **The whole lab costs less than 5 US cents** with the model below; the last cell shows your total.

## Setup

You need `pip install openai python-dotenv pydantic`. Create a file called `.env` in the `your-code` folder with one line, `OPENAI_API_KEY=sk-...` (the key Ironhack gave you). The repository's `.gitignore` already lists `.env`, so git will not see it. **Never write the key in the notebook**, and check with `git status` before you commit that `.env` is not listed.

In [3]:
pip install openai python-dotenv pydantic

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import json
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import Literal

pd.set_option("display.max_colwidth", 100)

load_dotenv()
print("Key found:", "OPENAI_API_KEY" in os.environ)
client = OpenAI()
MODEL = "gpt-6-luna"

PRICES = {"gpt-6-luna": (0.10, 0.50)}      # US dollars per million tokens (input, output), October 2026
usage_log = []                               # one row per API call, so you can add up the cost at the end

def track(response, step):
    """Record the tokens of a response under the name of a lab step, and return the response."""
    usage_log.append({"step": step, "input": response.usage.prompt_tokens, "output": response.usage.completion_tokens})
    return response

Key found: True


In [3]:
messages = pd.read_csv("../data/messages.csv")
examples = pd.read_csv("../data/examples.csv")
intents = sorted(messages["category"].unique())
print(messages.shape, examples.shape)
print(intents)

(60, 2) (20, 2)
['card_arrival', 'card_delivery_estimate', 'card_not_working', 'change_pin', 'compromised_card', 'exchange_rate', 'lost_or_stolen_card', 'pin_blocked', 'top_up_failed', 'top_up_reverted']


**Checkpoint:** `Key found: True`, then `(60, 2) (20, 2)` and the ten intents of yesterday's lab, from `card_arrival` to `top_up_reverted`.

## Challenge 1: a first call, and what it costs

1. Write a function `cost(input_tokens, output_tokens)` that returns the cost in US dollars with the prices in `PRICES[MODEL]`.
2. Send the first customer message (`messages["text"][0]`) to the model with a system prompt that makes it a polite bank support assistant answering in at most two sentences. Wrap the call in `track(..., "first call")`.
3. Print the reply, the number of input and output tokens, and the cost of the call.

In [7]:
# 1. Cost of a call
MODEL = "gpt-6-luna"

PRICES = {
    "gpt-6-luna": (0.10, 0.50)
}

def cost(input_tokens, output_tokens, model=MODEL):
    price_in, price_out = PRICES[model]
    return (input_tokens * price_in + output_tokens * price_out) / 1_000_000


def track(response, label):
    input_tokens = response.usage.input_tokens
    output_tokens = response.usage.output_tokens
    print(f"\Tracking {label}")
    print(f"Total Tokens: {input_tokens + output_tokens}")
    return response

# 2. First call
response = track(
    client.responses.create(
        model=MODEL,
        instructions=(
            "You are a polite bank support assistant. "
            "Answer customer questions clearly and professionally "
            "in at most two sentences."
        ),
        input=str(messages["text"].iloc[0])
    ),
    "first call"
)

# 3. Reply, tokens and cost
input_tokens = response.usage.input_tokens
output_tokens = response.usage.output_tokens
call_cost = cost(input_tokens, output_tokens)

print("\nCustomer message:", messages["text"].iloc[0])
print("Assistant reply:", response.output_text)
print("Input tokens:", input_tokens)
print("Output tokens:", output_tokens)
print(f"Cost: ${call_cost:.8f}")

\Tracking first call
Total Tokens: 195

Customer message: My card is being used online to make a bunch of purchases! Someone else is using it fraudulently. Can you help me freeze it?
Assistant reply: I’m sorry—that’s stressful. I can’t freeze the card from this chat, so use your banking app’s **Cards** section to freeze it now, then contact your bank using the number on its official website or the back of the card to report the purchases and request a replacement.
Input tokens: 59
Output tokens: 136
Cost: $0.00007390


**Checkpoint:** a short, polite reply, about 60 input and 50 output tokens, and a cost below a hundredth of a cent (about $0.00008). Read the reply: with no rules in the system prompt, the model may ask for details a bank should never ask for in a chat (in the reference run, it asked for the last four digits of the card and the billing postcode). Challenge 5 fixes that with constraints.

## Challenge 2: zero-shot intent classification

1. Write a system prompt that asks the model to classify a customer message into exactly one of the ten intents, lists the intents (you can join the list `intents` into the prompt), and asks for **the intent name only**.
2. Write `classify(text)`: one call with your system prompt and the message, `temperature=0`, tracked as `"zero-shot"`, returning the reply with spaces removed (`.strip()`).
3. Classify the 60 messages, store the predictions in a column `zero_shot`, and compute the accuracy.
4. Count the predictions that are **not** one of the ten intents.

In [11]:
# 1. System prompt
system_prompt = f"""
You are a banking customer support intent classifier.

Classify each customer message into exactly ONE of the following intents:
{", ".join(intents)}

Rules:
- Choose only one intent from the list.
- Return the intent name only.
- Do not provide explanations or additional text.
"""

# 2. Classifier
MODEL = "gpt-4.1-mini"

def track(response, label):
    usage = response.usage

    if hasattr(usage, "input_tokens"):
        input_tokens = usage.input_tokens
        output_tokens = usage.output_tokens
    else:
        input_tokens = usage.prompt_tokens
        output_tokens = usage.completion_tokens

    print(f"{label}: {input_tokens} input tokens, {output_tokens} output tokens")

    return response

def classify(text):
    response = track(
        client.chat.completions.create(
            model=MODEL,
            temperature=0,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": str(text)}
            ]
        ),
        "zero-shot"
    )

    return response.choices[0].message.content.strip()

messages["zero_shot"] = messages["text"].apply(classify)

# 3. Predictions and accuracy
accuracy = (messages["zero_shot"] == messages["category"]).mean()

print(f"Zero-shot accuracy: {accuracy:.2%}")


# 4. Invalid answers
invalid_predictions = (~messages["zero_shot"].isin(intents)).sum()

print(f"Invalid predictions: {invalid_predictions}")

zero-shot: 132 input tokens, 4 output tokens
zero-shot: 111 input tokens, 2 output tokens
zero-shot: 111 input tokens, 4 output tokens
zero-shot: 139 input tokens, 4 output tokens
zero-shot: 121 input tokens, 4 output tokens
zero-shot: 115 input tokens, 3 output tokens
zero-shot: 109 input tokens, 2 output tokens
zero-shot: 119 input tokens, 3 output tokens
zero-shot: 110 input tokens, 5 output tokens
zero-shot: 117 input tokens, 2 output tokens
zero-shot: 110 input tokens, 4 output tokens
zero-shot: 114 input tokens, 3 output tokens
zero-shot: 110 input tokens, 2 output tokens
zero-shot: 109 input tokens, 5 output tokens
zero-shot: 112 input tokens, 2 output tokens
zero-shot: 117 input tokens, 3 output tokens
zero-shot: 110 input tokens, 3 output tokens
zero-shot: 113 input tokens, 4 output tokens
zero-shot: 118 input tokens, 4 output tokens
zero-shot: 118 input tokens, 2 output tokens
zero-shot: 112 input tokens, 3 output tokens
zero-shot: 113 input tokens, 4 output tokens
zero-shot:

**Checkpoint:** an accuracy between about 0.8 and 0.9 (0.88 in the reference solution), and no answers outside the ten intents, if your prompt asks for the name only.

In [13]:
# 5. Show the messages that the zero-shot classifier got wrong, with the true and the predicted intent
wrong_predictions = messages[
    messages["category"] != messages["zero_shot"]
]

# Display the message, true intent, and predicted intent
display(
    wrong_predictions[["text", "category", "zero_shot"]]
    .rename(columns={
        "text": "Customer Message",
        "category": "True Intent",
        "zero_shot": "Predicted Intent"
    })
)

# Count incorrect predictions
print(f"Incorrect predictions: {len(wrong_predictions)} out of {len(messages)}")

,Customer Message,True Intent,Predicted Intent
7,Im very upset as my top-up was canceled and I have no idea why.,top_up_reverted,top_up_failed
10,How do I track my card?,card_arrival,card_delivery_estimate
11,Send my card as soon as you are able to.,card_delivery_estimate,card_arrival
30,How do I unblock my card using the app?,card_not_working,pin_blocked
36,How do I freeze my card using the app?,compromised_card,lost_or_stolen_card
52,When will my card arrive?,card_delivery_estimate,card_arrival
54,Causes of top-up cancellation,top_up_reverted,top_up_failed
56,How long should my new card take to arrive?,card_arrival,card_delivery_estimate


Incorrect predictions: 8 out of 60


**Question:** look at the mistakes. Which intents does the model confuse, and is it the same pattern as yesterday's models?

the mistakes are done on sematically similar intents. The subjects of the True intent vs the Predicted Intent are somewhat similar. for example to_up_reverted is similar to top_up_failed. Another example is card_arrival vs card_delivery_estimate. It seems that this model follows the same pattern as yesterday's models, where it finds difficulty in ascerting the exact topic.

## Challenge 3: few-shot classification

The intent names do not tell the model where the bank draws the line between, say, `card_arrival` and `card_delivery_estimate`. Examples can.

1. Build `few_shot_messages`: your system prompt, followed by the 20 rows of `examples` as pairs of messages, `{"role": "user", "content": text}` then `{"role": "assistant", "content": category}`.
2. Write `classify_few_shot(text)`, like `classify` but sending `few_shot_messages` plus the new message, tracked as `"few-shot"`.
3. Classify the 60 messages into a column `few_shot` and compute the accuracy.
4. From `usage_log`, compute the average number of input tokens per call for zero-shot and for few-shot, and the cost of classifying **1,000** messages with each.

In [14]:
# 1. Few-shot messages
few_shot_messages = [
    {"role": "system", "content": system_prompt}
]

for _, row in examples.iterrows():
    few_shot_messages.append({
        "role": "user",
        "content": str(row["text"])
    })
    few_shot_messages.append({
        "role": "assistant",
        "content": str(row["category"])
    })

print(f"Total messages in prompt: {len(few_shot_messages)}")

# 2. Classifier
def classify_few_shot(text):
    response = track(
        client.chat.completions.create(
            model=MODEL,
            temperature=0,
            messages=few_shot_messages + [
                {"role": "user", "content": str(text)}
            ]
        ),
        "few-shot"
    )

    return response.choices[0].message.content.strip()

# 3. Predictions and accuracy
messages["few_shot"] = messages["text"].apply(classify_few_shot)

few_shot_accuracy = (
    messages["few_shot"] == messages["category"]
).mean()

print(f"Few-shot accuracy: {few_shot_accuracy:.2%}")

# Counting invalid predictions
invalid_few_shot = (~messages["few_shot"].isin(intents)).sum()

print(f"Invalid predictions: {invalid_few_shot}")

# Displaying first 10 predictions
display(
    messages[["text", "category", "zero_shot", "few_shot"]].head(10)
)

# 4. Tokens per call and cost per 1,000 messages
usage_log = []

def track(response, label):
    usage = response.usage

    if hasattr(usage, "input_tokens"):
        input_tokens = usage.input_tokens
        output_tokens = usage.output_tokens
    else:
        input_tokens = usage.prompt_tokens
        output_tokens = usage.completion_tokens

    usage_log.append({
        "label": label,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "cost": cost(input_tokens, output_tokens, model=MODEL)
    })

    return response

Total messages in prompt: 41
few-shot: 598 input tokens, 4 output tokens
few-shot: 577 input tokens, 2 output tokens
few-shot: 577 input tokens, 4 output tokens
few-shot: 605 input tokens, 4 output tokens
few-shot: 587 input tokens, 4 output tokens
few-shot: 581 input tokens, 3 output tokens
few-shot: 575 input tokens, 2 output tokens
few-shot: 585 input tokens, 4 output tokens
few-shot: 576 input tokens, 5 output tokens
few-shot: 583 input tokens, 2 output tokens
few-shot: 576 input tokens, 3 output tokens
few-shot: 580 input tokens, 3 output tokens
few-shot: 576 input tokens, 2 output tokens
few-shot: 575 input tokens, 5 output tokens
few-shot: 578 input tokens, 2 output tokens
few-shot: 583 input tokens, 3 output tokens
few-shot: 576 input tokens, 3 output tokens
few-shot: 579 input tokens, 4 output tokens
few-shot: 584 input tokens, 4 output tokens
few-shot: 584 input tokens, 2 output tokens
few-shot: 578 input tokens, 3 output tokens
few-shot: 579 input tokens, 4 output tokens
few

,text,category,zero_shot,few_shot
0,My card is being used online to make a bunch of purchases! Someone else is using it fraudulently...,compromised_card,compromised_card,compromised_card
1,How do I find the exchange rate?,exchange_rate,exchange_rate,exchange_rate
2,The app reverted what I topped up.,top_up_reverted,top_up_reverted,top_up_reverted
3,What do I do to stop unauthorized transactions on my card? I was never in the place from which t...,compromised_card,compromised_card,compromised_card
4,How long does it take to get my card? Can I choose when to receive it?,card_delivery_estimate,card_delivery_estimate,card_delivery_estimate
5,What is the tracking number for my card that was mailed?,card_arrival,card_arrival,card_arrival
6,The exchange rate would be?,exchange_rate,exchange_rate,exchange_rate
7,Im very upset as my top-up was canceled and I have no idea why.,top_up_reverted,top_up_failed,top_up_reverted
8,I think my card was stolen.,lost_or_stolen_card,lost_or_stolen_card,lost_or_stolen_card
9,Please tell me which cash machines will allow me to change my pin.,change_pin,change_pin,change_pin


**Checkpoint:** few-shot accuracy clearly higher than zero-shot (0.97 against 0.88 in the reference solution). Each few-shot call sends about 600 input tokens instead of about 100, so classifying 1,000 messages costs about 13 cents instead of about 3. The output is the same few tokens in both cases.

**Question:** compare with yesterday's lab, where zero-shot DeBERTa scored about 0.76, semantic search about 0.92 and the fine-tuned DistilBERT about 0.94 on the same ten intents. When would you use the LLM, and when the fine-tuned model? Think about accuracy, cost, speed, labels and privacy.

I would choose the few-shot LLM when accuracy and flexibility matter most, especially for lower-volume or changing classification tasks. For high-volume, privacy-sensitive banking applications with stable intent categories, I would still consider fine-tuned DistilBERT.

However, on just 60 messages, the 97% versus 94% accuracy difference represents approximately two additional correct predictions. I would test both models on a larger independent dataset before concluding that the LLM is genuinely more accurate.

## Challenge 4: structured output

The routing system needs more than the intent: it needs to know how urgent a message is and whether a person must handle it, in a format it can read without surprises.

1. Write a Pydantic class `Ticket` with four fields: `intent` (a `Literal` of the ten intents), `urgency` (`Literal["low", "medium", "high"]`), `needs_human` (`bool`, with a `Field` description saying it is true when the customer reports fraud, a lost or stolen card, or is very upset) and `summary` (`str`, at most ten words).
2. Write `extract(text)` with `client.chat.completions.parse`, the few-shot messages of Challenge 3 as context, an extra system message asking for the ticket, and `response_format=Ticket`. Track it as `"structured"` and return the parsed `Ticket`.
3. Extract a ticket for each of the 60 messages, and build a DataFrame with the message, the true intent and the four fields.
4. Compute the accuracy of `intent`, and show how many tickets need a person, and the urgency counts.

In [21]:
MODEL = "gpt-4.1-mini"

PRICES = {
    "gpt-6-luna": (0.10, 0.50),
    "gpt-4.1-mini": (0.40, 1.60)
}

def cost(input_tokens, output_tokens, model=None):
    if model is None:
        model = MODEL

    price_in, price_out = PRICES[model]

    return (
        input_tokens * price_in +
        output_tokens * price_out
    ) / 1_000_000

In [24]:
# 1. Ticket
IntentLiteral = Literal.__getitem__(tuple(intents))

class Ticket(BaseModel):
    intent: IntentLiteral
    urgency: Literal["low", "medium", "high"]

    needs_human: bool = Field(
        description=(
            "True when the customer reports fraud, "
            "a lost or stolen card, or is very upset. "
            "Otherwise False."
        )
    )

    summary: str = Field(
        description="Summarize the customer's issue in at most ten words."
    )

# 2. Extraction
def extract(text):
    response = track(
        client.chat.completions.parse(
            model=MODEL,
            temperature=0,
            messages=few_shot_messages + [
                {
                    "role": "system",
                    "content": (
                        "Create a structured customer support ticket. "
                        "Choose exactly one of the allowed intents. "
                        "Set urgency to low, medium, or high. "
                        "Set needs_human to True if the customer reports "
                        "fraud, a lost or stolen card, or is very upset. "
                        "Write a summary of at most ten words."
                    )
                },
                {
                    "role": "user",
                    "content": str(text)
                }
            ],
            response_format=Ticket
        ),
        "structured"
    )

    ticket = response.choices[0].message.parsed

    if ticket is None:
        raise ValueError("No structured ticket was returned.")

    return ticket

# 3. Tickets for the 60 messages
tickets = messages["text"].apply(extract)

# Building the DataFrame
tickets_df = pd.DataFrame([
    {
        "message": text,
        "true_intent": true_intent,
        **ticket.model_dump()
    }
    for text, true_intent, ticket in zip(
        messages["text"],
        messages["category"],
        tickets
    )
])

display(tickets_df.head(10))

# 4. Accuracy, people needed, urgency
structured_accuracy = (
    tickets_df["intent"] == tickets_df["true_intent"]
).mean()

# Number of tickets requiring human assistance
human_tickets = tickets_df["needs_human"].sum()

# Distribution of urgency levels
urgency_counts = tickets_df["urgency"].value_counts()

print(f"Structured intent accuracy: {structured_accuracy:.2%}")
print(f"Tickets needing a human: {human_tickets}")
print("\nUrgency counts:")
print(urgency_counts)

,message,true_intent,intent,urgency,needs_human,summary
0,My card is being used online to make a bunch of purchases! Someone else is using it fraudulently...,compromised_card,compromised_card,high,True,"Fraudulent online purchases on card, needs freezing"
1,How do I find the exchange rate?,exchange_rate,exchange_rate,low,False,Customer wants to know how to find exchange rate
2,The app reverted what I topped up.,top_up_reverted,top_up_reverted,medium,False,Top up amount was reverted by the app
3,What do I do to stop unauthorized transactions on my card? I was never in the place from which t...,compromised_card,compromised_card,high,True,"Unauthorized transactions on card, customer requests help"
4,How long does it take to get my card? Can I choose when to receive it?,card_delivery_estimate,card_delivery_estimate,low,False,Inquiry about card delivery time and scheduling
5,What is the tracking number for my card that was mailed?,card_arrival,card_arrival,low,False,Customer requests tracking number for mailed card
6,The exchange rate would be?,exchange_rate,exchange_rate,low,False,Customer inquires about the exchange rate
7,Im very upset as my top-up was canceled and I have no idea why.,top_up_reverted,top_up_reverted,high,True,Customer upset about canceled top-up without explanation
8,I think my card was stolen.,lost_or_stolen_card,lost_or_stolen_card,high,True,Customer reports stolen card
9,Please tell me which cash machines will allow me to change my pin.,change_pin,change_pin,low,False,Customer asks where to change PIN at cash machines


Structured intent accuracy: 93.33%
Tickets needing a human: 17

Urgency counts:
urgency
medium    24
low       20
high      16
Name: count, dtype: int64


**Checkpoint:** an intent accuracy close to the few-shot one (0.95 in the reference solution), every field filled with an allowed value, and about a dozen tickets (13 of 60 in the reference) that need a person, mostly fraud and lost cards.

In [25]:
# 5. Which intents does the model send to a person most often? Show the share of needs_human per true intent.
display(
    tickets_df[tickets_df["needs_human"] == True]
)

,message,true_intent,intent,urgency,needs_human,summary
0,My card is being used online to make a bunch of purchases! Someone else is using it fraudulently...,compromised_card,compromised_card,high,True,"Fraudulent online purchases on card, needs freezing"
3,What do I do to stop unauthorized transactions on my card? I was never in the place from which t...,compromised_card,compromised_card,high,True,"Unauthorized transactions on card, customer requests help"
7,Im very upset as my top-up was canceled and I have no idea why.,top_up_reverted,top_up_reverted,high,True,Customer upset about canceled top-up without explanation
8,I think my card was stolen.,lost_or_stolen_card,lost_or_stolen_card,high,True,Customer reports stolen card
13,Some idiot stole my card.,lost_or_stolen_card,lost_or_stolen_card,high,True,Customer reports stolen card
21,"Someone has my card number, freeze my account.",compromised_card,compromised_card,high,True,"Customer reports card number compromised, requests account freeze"
26,I think my child used my card while I wasn't home.,compromised_card,compromised_card,high,True,Unauthorized card use suspected by child
27,My card doesn't accept any transaction at all. What's wrong??,card_not_working,card_not_working,high,True,Card not accepting any transactions
36,How do I freeze my card using the app?,compromised_card,compromised_card,high,True,Customer wants to freeze card via app
37,I believe my credit card was stolen.,lost_or_stolen_card,lost_or_stolen_card,high,True,Customer reports stolen credit card


**Question:** does `needs_human` follow the rule you gave it? Read a few tickets: would you trust the `urgency` and `needs_human` fields enough to route messages automatically? What would you check first?

According to the results, it seems that the model was able to correct;y identify most of the tickets that need human attention correctly according to the parameters given. I believe that the urgency level was correct for most of the tickets but not all of them, first i would check the true intent of the message. For example the message "My card doesn't accept any transaction at all. What's Wrong" could be solved by other means (checking the banking app) instead of requiring human attention

## Challenge 5: draft replies in a conversation

Finally, the assistant drafts replies for the agents. It must use the bank's real policies, and remember what the customer already said.

1. Write a system prompt with the role, the constraints (at most three sentences, never ask for the full card number, the PIN or passwords) and this context:
   - New cards arrive in 5 to 7 working days. After 10 working days, the customer can order a free replacement in the app under Card > Replace.
   - A lost or stolen card must be frozen immediately in the app under Card > Freeze; the bank then sends a new card for free.
   - A blocked PIN can be unblocked in the app under Card > PIN after confirming identity.
2. Write `reply(history, text)`: add the customer's message to `history`, call the model (tracked as `"conversation"`), add the answer as an `assistant` message, and return the answer.
3. Run this three-turn conversation and print each reply:
   - *Hi, I'm Marta. I ordered a new card 12 working days ago and it still hasn't come.*
   - *Can I still use my old card in the meantime?*
   - *Thanks. Remind me, how many days did I say I've been waiting?*

In [26]:
# 1. System prompt
conversation_prompt = """
You are a polite and professional customer support assistant for a bank.

Rules:
- Answer in at most three sentences.
- Never ask for the customer's full card number, PIN, or passwords.
- Use only the bank policies provided below.
- Do not invent policies or information.
- Remember information the customer has shared earlier in the conversation.
- If you do not know something, say so clearly.

Bank policies:
1. New cards arrive in 5 to 7 working days. After 10 working days, customers can order a free replacement in the app under Card > Replace.

2. A lost or stolen card must be frozen immediately in the app under Card > Freeze. The bank then sends a new card for free.

3. A blocked PIN can be unblocked in the app under Card > PIN after confirming identity.
"""

# 2. Reply with history
def reply(history, text):

    history.append({
        "role": "user",
        "content": text
    })

    # Sending the conversation history to the model
    response = track(
        client.chat.completions.create(
            model=MODEL,
            temperature=0,
            messages=[
                {"role": "system", "content": conversation_prompt}
            ] + history
        ),
        "conversation"
    )

    # Extracting the assistant's answer
    answer = response.choices[0].message.content.strip()

    # Saving the assistant's answer to the conversation history
    history.append({
        "role": "assistant",
        "content": answer
    })

    return answer

# 3. Three turns
history = []

customer_messages = [
    "Hi, I'm Marta. I ordered a new card 12 working days ago and it still hasn't come.",
    "Can I still use my old card in the meantime?",
    "Thanks. Remind me, how many days did I say I've been waiting?"
]

# Running the conversation
for message in customer_messages:

    print("Customer:", message)

    answer = reply(history, message)

    print("Assistant:", answer)
    print()


Customer: Hi, I'm Marta. I ordered a new card 12 working days ago and it still hasn't come.
Assistant: Hello Marta, since it has been more than 10 working days, you can order a free replacement card in the app under Card > Replace. If you need further assistance, please let me know.

Customer: Can I still use my old card in the meantime?
Assistant: If your old card is not lost, stolen, or frozen, you can continue using it until your new card arrives. If you have any concerns about its security, please consider freezing it in the app under Card > Freeze.

Customer: Thanks. Remind me, how many days did I say I've been waiting?
Assistant: You mentioned that you have been waiting for 12 working days for your new card.



**Checkpoint:** three short replies. The first one uses the policy (more than 10 working days, so a free replacement under Card > Replace) and the customer's name. The third proves the history works: it remembers the 12 working days. No reply asks for the PIN or the card number.

**Question:** the second question (can I use my old card?) is not covered by the policies you gave. What did the assistant answer, and why is that a risk for the bank? How would you reduce it?

- The asistant answered: "If your old card is not lost, stolen, or frozen, you can continue using it until your new card arrives. If you have any concerns about its security, please consider freezing it in the app under Card > Freeze."
- This is a hallucination from the model. The risk is that the customer might rely on incorrect information, leading to increased complaints, even liability from the bank. The model made an unsupported statement about card functionality, despite being instructed to use only the bank's policies.

- To reduce it, I would instruct the model to answer only using verified bank policies and never make assumptions. Also, if the policies do not cover a question it should refer it to a human agent.


## What did it cost?

Run the cell below to see the tokens and the cost of every step of the lab.

In [29]:
log = pd.DataFrame(usage_log)
summary = log.groupby("label", sort=False).agg(
    calls=("input_tokens", "count"),
    input_tokens=("input_tokens", "sum"),
    output_tokens=("output_tokens", "sum"),
    cost_usd=("cost", "sum")
)

print(f"Total: {len(log)} calls, ${summary['cost_usd'].sum():.4f}")
display(summary.round(5))

Total: 64 calls, $0.0243


,calls,input_tokens,output_tokens,cost_usd
label,,,,
structured,61,52773,1723,0.02387
conversation,3,781,102,0.00048


**Checkpoint:** about 180 calls and a total of about 2 cents (well under 5). The structured step is the most expensive, because it sends the 20 examples with every call; the zero-shot classifier is the cheapest.

## Extra: let the assistant check the delivery status

Agents can look up where a customer's card is. Give the assistant the same ability with **function calling**.

1. Write `get_card_status(customer_id)` that looks the customer up in the dictionary `CARD_ORDERS` below and returns a dictionary with the status, or an error if the customer is unknown.
2. Describe it in a `tools` list (one required string parameter, `customer_id`, with `"strict": True`).
3. Run a conversation with the support prompt where the customer writes *My customer id is C-1043, where is my new card?*: when the model asks for the tool, run it, send the result back with the role `tool`, and print the final answer. Track every call as `"extra"`.
4. Try it with the unknown customer `C-9999`. Does the assistant handle the error well?

In [32]:
CARD_ORDERS = {
    "C-1043": {"ordered": "2026-10-12", "status": "shipped", "carrier": "Correos", "expected": "2026-10-23"},
    "C-2210": {"ordered": "2026-10-20", "status": "being printed", "carrier": None, "expected": "2026-10-29"},
}

# Your code here
def get_card_status(customer_id):
    if customer_id in CARD_ORDERS:
        return {
            "customer_id": customer_id,
            **CARD_ORDERS[customer_id]
        }
    else:
        return {
            "error": "Customer not found",
            "customer_id": customer_id
        }

#defining tool schema
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_card_status",
            "description": (
                "Look up the current card order and delivery status "
                "for a customer using their customer ID."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "customer_id": {
                        "type": "string",
                        "description": "Customer ID, e.g. C-1043"
                    }
                },
                "required": ["customer_id"],
                "additionalProperties": False
            },
            "strict": True
        }
    }
]


#Conversation function
def card_support(customer_message):

    # Start the conversation with the existing bank support prompt
    history = [
        {"role": "system", "content": conversation_prompt},
        {
            "role": "system",
            "content": (
                "Use get_card_status to check card delivery questions. "
                "Never invent order statuses or delivery dates. "
                "If the customer is unknown, explain that the order "
                "could not be found and suggest contacting support. "
                "Do not claim that the customer's identity has been verified."
            )
        },
        {"role": "user", "content": customer_message}
    ]

    # First API call: model decides whether to use the tool
    response = track(
        client.chat.completions.create(
            model=MODEL,
            temperature=0,
            messages=history,
            tools=tools,
            tool_choice="auto"
        ),
        "extra"
    )

    assistant_message = response.choices[0].message
    history.append(assistant_message.model_dump(exclude_none=True))

    # Execute any tool calls
    for tool_call in assistant_message.tool_calls or []:

        if tool_call.function.name != "get_card_status":
            raise ValueError("Unexpected tool requested")

        arguments = json.loads(tool_call.function.arguments)

        result = get_card_status(arguments["customer_id"])

        print("Tool result:", result)

        # Send the tool result back into the conversation
        history.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": json.dumps(result)
        })

    # Second API call if the model used the tool
    if assistant_message.tool_calls:

        final_response = track(
            client.chat.completions.create(
                model=MODEL,
                temperature=0,
                messages=history
            ),
            "extra"
        )

        final_answer = final_response.choices[0].message.content
    else:
        final_answer = assistant_message.content

    print("\nAssistant:", final_answer)

    return final_answer


In [31]:
print(get_card_status("C-1043"))
print(get_card_status("C-9999"))

{'customer_id': 'C-1043', 'ordered': '2026-10-12', 'status': 'shipped', 'carrier': 'Correos', 'expected': '2026-10-23'}
{'error': 'Customer not found', 'customer_id': 'C-9999'}


In [33]:
answer = card_support(
    "My customer id is C-1043, where is my new card?"
)

Tool result: {'customer_id': 'C-1043', 'ordered': '2026-10-12', 'status': 'shipped', 'carrier': 'Correos', 'expected': '2026-10-23'}

Assistant: Your new card was shipped via Correos and is expected to arrive by October 23, 2026. If you do not receive it by then, please let us know.


In [34]:
answer_unknown = card_support(
    "My customer id is C-9999, where is my new card?"
)

Tool result: {'error': 'Customer not found', 'customer_id': 'C-9999'}

Assistant: We could not find an order for customer ID C-9999 in our system. Please double-check the ID or contact our support team for further assistance.


**Conclusions:** Tool calling reduces hallucination risk because the model obtains card-order information from a structured source rather than generating it from its general knowledge. However,knowing a customer ID is not proof of identity. There need to be implemented a better system for identification proof.

